# Curriculum NESTFUL eval — Colab + Google Drive

Multi-turn evaluace **baseline + LoRA checkpointů** (`curricullum/evaluation/run_eval.py`).
Stejný protokol jako `nestful_evaluation/run.py` → výstupy pro `eval_viewer.html`.

## Co nahrát na Google Drive

Kořen projektu (doporučená cesta): **`MyDrive/Tool-R0/`**

```
Tool-R0/
├── grpo_processing.py
├── nestful_evaluation/
│   └── run.py
├── curricullum/
│   ├── checkpoints/qwen3_4b_lora_grpo/
│   │   ├── stage1_1call/
│   │   │   ├── adapter_config.json
│   │   │   ├── adapter_model.safetensors   ← povinné (~100–200 MB / stage)
│   │   │   ├── tokenizer.json
│   │   │   ├── tokenizer_config.json
│   │   │   └── chat_template.jinja
│   │   ├── stage2_2call/   (stejné soubory)
│   │   └── stage3_3call/   (stejné soubory)
│   └── evaluation/
│       └── run_eval.py
└── eval_viewer.html          ← volitelné (prohlížení doma)
```

**Nenahrávej:** celý HF base model (stáhne se z Hubu), IBM NESTFUL repo (klonuje Colab), dataset `ibm-research/nestful` (HF).

**Tip:** Z DGX zkopíruj jen složky `stage*_*/` včetně `adapter_model.safetensors`. Bez adapter weights merge selže.

## Postup

1. Nahraj soubory na Drive (viz strom výše)
2. **Runtime → Change runtime type → A100** (T4 funguje jen s pilotem + nižším `MAX_MODEL_LEN`)
3. Uprav `DRIVE_PROJECT_DIR` v konfiguraci
4. **Run all**

In [ ]:
# ============ KONFIGURACE ============
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/Tool-R0"
BASE_MODEL = "Qwen/Qwen3-4B-Instruct-2507"

# Profily: baseline + stage1_1call + stage2_2call + stage3_3call
# Prázdné = všechny. Pilot: ONLY = ["baseline"]
ONLY = []  # např. ["baseline", "stage3_3call"]
SKIP_EXISTING = True  # přeskoč profily, které už mají summary na Drive

MAX_TASKS = 100       # pilot: 100, full: None (1861)
NUM_ROLLOUTS = 2      # full benchmark: 8
MAX_STEPS = 10
TEMPERATURE = 0.7
TOP_P = 0.95
MAX_NEW_TOKENS = 2048
MAX_MODEL_LEN = 12288  # T4: sniž na 8192 nebo 4096
GPU_MEMORY_UTILIZATION = 0.85
IBM_CALL_TIMEOUT = 30
ADVANCE_LOG_EVERY = 50
SEED = 0
HF_TOKEN = None  # nebo Colab secret HF_TOKEN

In [ ]:
import os, subprocess, sys, shutil

def sh(cmd, check=True):
    print(f"$ {cmd}")
    subprocess.run(cmd, shell=True, check=check)

from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = DRIVE_PROJECT_DIR.rstrip("/")
LOCAL_ROOT = "/content/Tool-R0"
NESTFUL_REPO_DIR = "/content/nestful_repo"

REQUIRED = [
    "grpo_processing.py",
    "nestful_evaluation/run.py",
    "curricullum/evaluation/run_eval.py",
    "curricullum/checkpoints/qwen3_4b_lora_grpo/stage1_1call/adapter_config.json",
    "curricullum/checkpoints/qwen3_4b_lora_grpo/stage1_1call/adapter_model.safetensors",
]
missing = [p for p in REQUIRED if not os.path.isfile(os.path.join(DRIVE_ROOT, p))]
if missing:
    raise FileNotFoundError(
        "Na Drive chybí:\n  " + "\n  ".join(missing)
    )

# Rychlé skripty lokálně; checkpointy + výsledky na Drive (symlink)
sh(f"rm -rf {LOCAL_ROOT}")
os.makedirs(LOCAL_ROOT, exist_ok=True)

for rel in ["grpo_processing.py", "nestful_evaluation/run.py", "curricullum/evaluation/run_eval.py"]:
    src = os.path.join(DRIVE_ROOT, rel)
    dst = os.path.join(LOCAL_ROOT, rel)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copy2(src, dst)

def link_drive(rel_path: str) -> None:
    src = os.path.join(DRIVE_ROOT, rel_path)
    dst = os.path.join(LOCAL_ROOT, rel_path)
    os.makedirs(src, exist_ok=True)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if os.path.islink(dst) or os.path.exists(dst):
        if os.path.islink(dst):
            os.unlink(dst)
        elif os.path.isdir(dst):
            pass  # already there
        else:
            os.remove(dst)
    if not os.path.exists(dst):
        os.symlink(src, dst)

link_drive("curricullum/checkpoints/qwen3_4b_lora_grpo")
link_drive("curricullum/evaluation/results")
link_drive("curricullum/evaluation/prepared")

OUTPUT_DIR = os.path.join(DRIVE_ROOT, "curricullum/evaluation/results")
print(f"Drive root:  {DRIVE_ROOT}")
print(f"Local root:  {LOCAL_ROOT}")
print(f"Výstupy:     {OUTPUT_DIR}")

In [ ]:
sh("nvidia-smi || true", check=False)
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Zapni GPU: Runtime → Change runtime type → A100")
print(f"GPU: {torch.cuda.get_device_name(0)}")
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"VRAM: {vram_gb:.1f} GB")
if vram_gb >= 35:
    MAX_MODEL_LEN = max(MAX_MODEL_LEN, 12288)
    GPU_MEMORY_UTILIZATION = min(GPU_MEMORY_UTILIZATION, 0.90)
elif vram_gb >= 20:
    MAX_MODEL_LEN = min(MAX_MODEL_LEN, 8192)
    GPU_MEMORY_UTILIZATION = min(GPU_MEMORY_UTILIZATION, 0.85)
else:
    MAX_MODEL_LEN = min(MAX_MODEL_LEN, 4096)
    GPU_MEMORY_UTILIZATION = min(GPU_MEMORY_UTILIZATION, 0.75)
print(f"→ MAX_MODEL_LEN={MAX_MODEL_LEN}, gpu_mem={GPU_MEMORY_UTILIZATION}")

In [ ]:
# vLLM pro Colab CUDA 12.x (cu129 wheel)
import gc, platform, subprocess, sys, textwrap, importlib
import torch
from packaging import version

PY = sys.executable
VLLM_VER = "0.22.0"
TORCH_CUDA = torch.version.cuda or ""
ARCH = "aarch64" if platform.machine() == "aarch64" else "x86_64"

def pip(*args, check=False):
    cmd = [PY, "-m", "pip", *args]
    print("$", " ".join(cmd))
    return subprocess.run(cmd, check=check)

def vllm_wheel_url(cu_tag: str) -> str:
    return (
        f"https://github.com/vllm-project/vllm/releases/download/v{VLLM_VER}/"
        f"vllm-{VLLM_VER}+cu{cu_tag}-cp38-abi3-manylinux_2_28_{ARCH}.whl"
    )

pip("install", "-q", "-U", "pip", "packaging", "datasets>=3.5.0", "huggingface_hub>=0.30.0")
pip("install", "-q", "-U", "transformers", "peft", "accelerate", "safetensors", "torchao>=0.16.0")

if TORCH_CUDA.startswith("13"):
    pip("install", "-q", "-U", "vllm")
else:
    cu_tag = "129"
    wheel = vllm_wheel_url(cu_tag)
    pt_index = f"https://download.pytorch.org/whl/cu{cu_tag}"
    pip("uninstall", "-y", "vllm", check=False)
    rc = pip("install", "-q", wheel, "--extra-index-url", pt_index).returncode
    if rc != 0:
        pip("install", "-q", "vllm", "--extra-index-url", pt_index)

import vllm
print(f"vllm {vllm.__version__}")

from vllm import LLM
from transformers import AutoConfig
cfg = AutoConfig.from_pretrained(BASE_MODEL, trust_remote_code=True)
print(f"AutoConfig model_type={cfg.model_type} OK")

_test = textwrap.dedent(f"""
import gc, torch
from vllm import LLM
llm = LLM(model={BASE_MODEL!r}, tensor_parallel_size=1,
          gpu_memory_utilization={GPU_MEMORY_UTILIZATION},
          max_model_len={MAX_MODEL_LEN}, enforce_eager=True,
          trust_remote_code=True)
del llm
gc.collect(); torch.cuda.empty_cache()
print("Qwen3 smoke test OK")
""")
r = subprocess.run([PY, "-c", _test], capture_output=True, text=True)
print(r.stdout or r.stderr[-4000:])
if r.returncode != 0:
    raise RuntimeError("vLLM smoke test selhal")
VLLM_SMOKE_OK = True
print("✓ Instalace OK")

In [ ]:
from huggingface_hub import login
if HF_TOKEN:
    login(token=HF_TOKEN)
else:
    try:
        from google.colab import userdata
        login(token=userdata.get("HF_TOKEN"))
    except Exception:
        login()

In [ ]:
SENTINEL = "/content/nestful_repo/data_v2/executable_functions/func_file_map.json"
if not os.path.isfile(SENTINEL):
    sh("rm -rf /content/nestful_repo")
    sh("git clone --depth 1 https://github.com/IBM/NESTFUL.git /content/nestful_repo")
else:
    print("IBM repo OK")

from datasets import load_dataset
load_dataset("ibm-research/nestful", split="train[:1]")
print("HF dataset OK")

In [ ]:
# ============ CURRICULUM EVAL (baseline + checkpoints) ============
if not globals().get("VLLM_SMOKE_OK"):
    raise RuntimeError("Nejdřív spusť instalaci + smoke test!")

# Smaž neúplné merge z předchozího selhání (torchao apod.)
import glob
for bad in glob.glob(os.path.join(DRIVE_ROOT, "curricullum/evaluation/prepared/curriculum_stage*")):
    if not os.path.isfile(os.path.join(bad, ".merge_done")):
        print(f"Removing incomplete prepared dir: {bad}")
        shutil.rmtree(bad, ignore_errors=True)

RUN_EVAL = os.path.join(LOCAL_ROOT, "curricullum/evaluation/run_eval.py")
cmd = [
    sys.executable, "-u", RUN_EVAL,
    "--base-model", BASE_MODEL,
    "--output-dir", "curricullum/evaluation/results",
    "--prepared-root", "curricullum/evaluation/prepared",
    "--num-rollouts", str(NUM_ROLLOUTS),
    "--max-steps", str(MAX_STEPS),
    "--temperature", str(TEMPERATURE),
    "--top-p", str(TOP_P),
    "--max-new-tokens", str(MAX_NEW_TOKENS),
    "--max-model-len", str(MAX_MODEL_LEN),
    "--tensor-parallel-size", "1",
    "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),
    "--ibm-call-timeout", str(IBM_CALL_TIMEOUT),
    "--advance-log-every", str(ADVANCE_LOG_EVERY),
    "--seed", str(SEED),
    "--nestful-repo-dir", NESTFUL_REPO_DIR,
]
if MAX_TASKS is not None:
    cmd.extend(["--max-tasks", str(MAX_TASKS)])
if SKIP_EXISTING:
    cmd.append("--skip-existing")
if ONLY:
    cmd.extend(["--only", *ONLY])

n_tasks = MAX_TASKS if MAX_TASKS is not None else 1861
n_profiles = len(ONLY) if ONLY else 4
print(f"Profily: {ONLY or ['baseline','stage1_1call','stage2_2call','stage3_3call']}")
print(f"Eval: ~{n_profiles} modelů × {n_tasks} úloh × {NUM_ROLLOUTS} rolloutů")
print(f"Výstupy na Drive: {OUTPUT_DIR}")
print(" ".join(cmd), "\n")

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
proc = subprocess.Popen(
    cmd, cwd=LOCAL_ROOT,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, env=env,
)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end="", flush=True)
if proc.wait() != 0:
    raise RuntimeError("run_eval.py selhal — scrolluj nahoru pro Traceback")
print("\n✓ Curriculum eval dokončena")

In [ ]:
import json, glob

print(f"=== Soubory v {OUTPUT_DIR} ===")
for path in sorted(glob.glob(os.path.join(OUTPUT_DIR, "*"))):
    sz = os.path.getsize(path) / 1e6
    print(f"  {os.path.basename(path)}  ({sz:.1f} MB)")

summaries = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*_multiturn_summary.json")))
if summaries:
    print("\n=== Přehled profilů ===")
    for sp in summaries:
        with open(sp) as f:
            s = json.load(f)
        prof = s.get("model_profile", os.path.basename(sp))
        acc = s.get("final_answer_accuracy_percent", s.get("mean_score_percent", "?"))
        print(f"  {prof}: pass={s.get('passed')}/{s.get('total_rollouts')} ({acc}%)")

print("\nStáhni *_multiturn_predictions.jsonl a otevři eval_viewer.html lokálně.")